# AutoML Baseline against Entity Embeddings of Categorical Variables
This Notebooks shall demonstrate 2 things
1. That complex customized approaches can be matched by much easier to use AutoML Frameworks. Hence AutoML Frameworks should be used as Baselines.
2. How such Baseline Comparisons ought to be conducted

In [4]:
!pip install scikit-learn xgboost tensorflow keras jupyter matplotlib pandas autogluon
!pip install kaggle

Start by downloading and uploading the following two files from Kaggle
- `train.csv`
- `store.csv`
And `store_states.csv` from the [original repository](https://github.com/entron/entity-embedding-rossmann/blob/master/store_states.csv).

The following code should download it automatically but in case of problems manual [download](https://www.kaggle.com/competitions/rossmann-store-sales/data) is possbile too (place extracted files into a directory called `content`)


In [ ]:
# Download Extra Files from original Repo & Competition
!kaggle competitions download -c rossmann-store-sales
!unzip -q rossmann-store-sales.zip -d ./content


import urllib.request

url = "https://raw.githubusercontent.com/entron/entity-embedding-rossmann/master/store_states.csv"
urllib.request.urlretrieve(url, "content/store_states.csv")

In [8]:
import numpy
import pickle
import csv
from datetime import datetime
from sklearn import preprocessing
import numpy as np
import random
from sklearn.dummy import DummyRegressor
random.seed(42)

my_results = {}

def mape(y, prediction):
  relative_err = numpy.absolute((y - prediction) / y)
  return numpy.sum(relative_err) / len(y)

train_data = "./content/train.csv"
store_data = "./content/store.csv"
store_states = './content/store_states.csv'

In [2]:
# extract_csv_files.py
def csv2dicts(csvfile):
    data = []
    keys = []
    for row_index, row in enumerate(csvfile):
        if row_index == 0:
            keys = row
            print(row)
            continue
        # if row_index % 10000 == 0:
        #     print(row_index)
        data.append({key: value for key, value in zip(keys, row)})
    return data


def set_nan_as_string(data, replace_str='0'):
    for i, x in enumerate(data):
        for key, value in x.items():
            if value == '':
                x[key] = replace_str
        data[i] = x

with open(train_data) as csvfile:
    data = csv.reader(csvfile, delimiter=',')
    with open('train_data.pickle', 'wb') as f:
        data = csv2dicts(data)
        data = data[::-1]
        pickle.dump(data, f, -1)
        print(data[:3])


with open(store_data) as csvfile, open(store_states) as csvfile2:
    data = csv.reader(csvfile, delimiter=',')
    state_data = csv.reader(csvfile2, delimiter=',')
    with open('store_data.pickle', 'wb') as f:
        data = csv2dicts(data)
        state_data = csv2dicts(state_data)
        set_nan_as_string(data)
        for index, val in enumerate(data):
            state = state_data[index]
            val['State'] = state['State']
            data[index] = val
        pickle.dump(data, f, -1)
        print(data[:2])

['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']
[{'Store': '1115', 'DayOfWeek': '2', 'Date': '2013-01-01', 'Sales': '0', 'Customers': '0', 'Open': '0', 'Promo': '0', 'StateHoliday': 'a', 'SchoolHoliday': '1'}, {'Store': '1114', 'DayOfWeek': '2', 'Date': '2013-01-01', 'Sales': '0', 'Customers': '0', 'Open': '0', 'Promo': '0', 'StateHoliday': 'a', 'SchoolHoliday': '1'}, {'Store': '1113', 'DayOfWeek': '2', 'Date': '2013-01-01', 'Sales': '0', 'Customers': '0', 'Open': '0', 'Promo': '0', 'StateHoliday': 'a', 'SchoolHoliday': '1'}]
['Store', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']
['Store', 'State']
[{'Store': '1', 'StoreType': 'c', 'Assortment': 'a', 'CompetitionDistance': '1270', 'CompetitionOpenSinceMonth': '9', 'CompetitionOpenSinceYear': '2008', 'Promo2': '0', 'Promo2SinceWeek': '0', 'Promo2SinceYear': 

In [3]:
# prepare_features.py
with open('train_data.pickle', 'rb') as f:
    train_data = pickle.load(f)
    num_records = len(train_data)
with open('store_data.pickle', 'rb') as f:
    store_data = pickle.load(f)


def feature_list(record):
    dt = datetime.strptime(record['Date'], '%Y-%m-%d')
    store_index = int(record['Store'])
    year = dt.year
    month = dt.month
    day = dt.day
    day_of_week = int(record['DayOfWeek'])
    try:
        store_open = int(record['Open'])
    except:
        store_open = 1

    promo = int(record['Promo'])

    return [store_open,
            store_index,
            day_of_week,
            promo,
            year,
            month,
            day,
            store_data[store_index - 1]['State']
            ]


train_data_X = []
train_data_y = []

for record in train_data:
    if record['Sales'] != '0' and record['Open'] != '':
        fl = feature_list(record)
        train_data_X.append(fl)
        train_data_y.append(int(record['Sales']))
print("Number of train datapoints: ", len(train_data_y))

print(min(train_data_y), max(train_data_y))

full_X = train_data_X
full_X = np.array(full_X)
train_data_X = np.array(train_data_X)
les = []
for i in range(train_data_X.shape[1]):
    le = preprocessing.LabelEncoder()
    le.fit(full_X[:, i])
    les.append(le)
    train_data_X[:, i] = le.transform(train_data_X[:, i])

with open('les.pickle', 'wb') as f:
    pickle.dump(les, f, -1)

train_data_X = train_data_X.astype(int)
train_data_y = np.array(train_data_y)

with open('feature_train_data.pickle', 'wb') as f:
    pickle.dump((train_data_X, train_data_y), f, -1)
    print(train_data_X[0], train_data_y[0])

Number of train datapoints:  844338
46 41551
[  0 109   1   0   0   0   0   7] 5961


In [4]:
# @title
# models.py
import numpy
numpy.random.seed(123)
from sklearn import linear_model
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn import neighbors
from sklearn.preprocessing import Normalizer

from keras.models import Sequential
from keras.models import Model as KerasModel
from keras.layers import Input, Dense, Activation, Reshape
from keras.layers import Concatenate
from tensorflow.keras.layers import Embedding
from keras.callbacks import ModelCheckpoint

import pickle


def embed_features(X, saved_embeddings_fname):
    # f_embeddings = open("embeddings_shuffled.pickle", "rb")
    f_embeddings = open(saved_embeddings_fname, "rb")
    embeddings = pickle.load(f_embeddings)

    index_embedding_mapping = {1: 0, 2: 1, 4: 2, 5: 3, 6: 4, 7: 5}
    X_embedded = []

    (num_records, num_features) = X.shape
    for record in X:
        embedded_features = []
        for i, feat in enumerate(record):
            feat = int(feat)
            if i not in index_embedding_mapping.keys():
                embedded_features += [feat]
            else:
                embedding_index = index_embedding_mapping[i]
                embedded_features += embeddings[embedding_index][feat].tolist()

        X_embedded.append(embedded_features)

    return numpy.array(X_embedded)


def split_features(X):
    X_list = []

    store_index = X[..., [1]]
    X_list.append(store_index)

    day_of_week = X[..., [2]]
    X_list.append(day_of_week)

    promo = X[..., [3]]
    X_list.append(promo)

    year = X[..., [4]]
    X_list.append(year)

    month = X[..., [5]]
    X_list.append(month)

    day = X[..., [6]]
    X_list.append(day)

    State = X[..., [7]]
    X_list.append(State)

    return X_list


class Model(object):

    def evaluate(self, X_val, y_val):
        assert(min(y_val) > 0)
        guessed_sales = self.guess(X_val)
        relative_err = numpy.absolute((y_val - guessed_sales) / y_val)
        result = numpy.sum(relative_err) / len(y_val)
        return result


class LinearModel(Model):

    def __init__(self, X_train, y_train, X_val, y_val):
        super().__init__()
        self.clf = linear_model.LinearRegression()
        self.clf.fit(X_train, numpy.log(y_train))
        print("Result on validation data: ", self.evaluate(X_val, y_val))

    def guess(self, feature):
        return numpy.exp(self.clf.predict(feature))


class RF(Model):

    def __init__(self, X_train, y_train, X_val, y_val):
        super().__init__()
        self.clf = RandomForestRegressor(n_estimators=200, verbose=True, max_depth=35, min_samples_split=2,
                                         min_samples_leaf=1)
        self.clf.fit(X_train, numpy.log(y_train))
        print("Result on validation data: ", self.evaluate(X_val, y_val))

    def guess(self, feature):
        return numpy.exp(self.clf.predict(feature))


class SVM(Model):

    def __init__(self, X_train, y_train, X_val, y_val):
        super().__init__()
        self.X_train = X_train
        self.y_train = y_train
        self.__normalize_data()
        self.clf = SVR(kernel='linear', degree=3, gamma='auto', coef0=0.0, tol=0.001,
                       C=1.0, epsilon=0.1, shrinking=True, cache_size=200, verbose=False, max_iter=-1)

        self.clf.fit(self.X_train, numpy.log(self.y_train))
        print("Result on validation data: ", self.evaluate(X_val, y_val))

    def __normalize_data(self):
        self.scaler = StandardScaler()
        self.X_train = self.scaler.fit_transform(self.X_train)

    def guess(self, feature):
        return numpy.exp(self.clf.predict(feature))


class XGBoost(Model):

    def __init__(self, X_train, y_train, X_val, y_val):
        super().__init__()
        dtrain = xgb.DMatrix(X_train, label=numpy.log(y_train))
        evallist = [(dtrain, 'train')]
        param = {'nthread': -1,
                 'max_depth': 7,
                 'eta': 0.02,
                 'silent': 1,
                 'objective': 'reg:linear',
                 'colsample_bytree': 0.7,
                 'subsample': 0.7}
        num_round = 3000
        self.bst = xgb.train(param, dtrain, num_round, evallist)
        print("Result on validation data: ", self.evaluate(X_val, y_val))

    def guess(self, feature):
        dtest = xgb.DMatrix(feature)
        return numpy.exp(self.bst.predict(dtest))


class HistricalMedian(Model):

    def __init__(self, X_train, y_train, X_val, y_val):
        super().__init__()
        self.history = {}
        self.feature_index = [1, 2, 3, 4]
        for x, y in zip(X_train, y_train):
            key = tuple(x[self.feature_index])
            self.history.setdefault(key, []).append(y)
        print("Result on validation data: ", self.evaluate(X_val, y_val))

    def guess(self, features):
        features = numpy.array(features)
        features = features[:, self.feature_index]
        guessed_sales = [numpy.median(self.history[tuple(feature)]) for feature in features]
        return numpy.array(guessed_sales)


class KNN(Model):

    def __init__(self, X_train, y_train, X_val, y_val):
        super().__init__()
        self.normalizer = Normalizer()
        self.normalizer.fit(X_train)
        self.clf = neighbors.KNeighborsRegressor(n_neighbors=10, weights='distance', p=1)
        self.clf.fit(self.normalizer.transform(X_train), numpy.log(y_train))
        print("Result on validation data: ", self.evaluate(self.normalizer.transform(X_val), y_val))

    def guess(self, feature):
        return numpy.exp(self.clf.predict(self.normalizer.transform(feature)))

class MeanModel(Model):

    def __init__(self, X_train, y_train, X_val, y_val):
        super().__init__()
        self.model = DummyRegressor(strategy="mean")
        self.model.fit(X_train, y_train)
        print("Result on validation data: ", self.evaluate(X_val, y_val))

    def guess(self, feature):
        return self.model.predict(feature)


class NN_with_EntityEmbedding(Model):

    def __init__(self, X_train, y_train, X_val, y_val):
        super().__init__()
        self.epochs = 10
        self.checkpointer = ModelCheckpoint(filepath="best_model_weights.keras", verbose=1, save_best_only=True)
        self.max_log_y = max(numpy.max(numpy.log(y_train)), numpy.max(numpy.log(y_val)))
        self.__build_keras_model()
        self.fit(X_train, y_train, X_val, y_val)

    def preprocessing(self, X):
        X_list = split_features(X)
        return X_list

    def __build_keras_model(self):
        input_store = Input(shape=(1,))
        output_store = Embedding(1115, 10, name='store_embedding')(input_store)
        output_store = Reshape(target_shape=(10,))(output_store)

        input_dow = Input(shape=(1,))
        output_dow = Embedding(7, 6, name='dow_embedding')(input_dow)
        output_dow = Reshape(target_shape=(6,))(output_dow)

        input_promo = Input(shape=(1,))
        output_promo = Dense(1)(input_promo)

        input_year = Input(shape=(1,))
        output_year = Embedding(3, 2, name='year_embedding')(input_year)
        output_year = Reshape(target_shape=(2,))(output_year)

        input_month = Input(shape=(1,))
        output_month = Embedding(12, 6, name='month_embedding')(input_month)
        output_month = Reshape(target_shape=(6,))(output_month)

        input_day = Input(shape=(1,))
        output_day = Embedding(31, 10, name='day_embedding')(input_day)
        output_day = Reshape(target_shape=(10,))(output_day)

        input_germanstate = Input(shape=(1,))
        output_germanstate = Embedding(12, 6, name='state_embedding')(input_germanstate)
        output_germanstate = Reshape(target_shape=(6,))(output_germanstate)

        input_model = [input_store, input_dow, input_promo,
                       input_year, input_month, input_day, input_germanstate]

        output_embeddings = [output_store, output_dow, output_promo,
                             output_year, output_month, output_day, output_germanstate]

        output_model = Concatenate()(output_embeddings)
        output_model = Dense(1000, kernel_initializer="uniform")(output_model)
        output_model = Activation('relu')(output_model)
        output_model = Dense(500, kernel_initializer="uniform")(output_model)
        output_model = Activation('relu')(output_model)
        output_model = Dense(1)(output_model)
        output_model = Activation('sigmoid')(output_model)

        self.model = KerasModel(inputs=input_model, outputs=output_model)

        self.model.compile(loss='mean_absolute_error', optimizer='adam')

    def _val_for_fit(self, val):
        val = numpy.log(val) / self.max_log_y
        return val

    def _val_for_pred(self, val):
        return numpy.exp(val * self.max_log_y)

    def fit(self, X_train, y_train, X_val, y_val):
        self.model.fit(self.preprocessing(X_train), self._val_for_fit(y_train),
                       validation_data=(self.preprocessing(X_val), self._val_for_fit(y_val)),
                       epochs=self.epochs, batch_size=128,
                       # callbacks=[self.checkpointer],
                       )
        # self.model.load_weights('best_model_weights.hdf5')
        print("Result on validation data: ", self.evaluate(X_val, y_val))

    def guess(self, features):
        features = self.preprocessing(features)
        result = self.model.predict(features).flatten()
        return self._val_for_pred(result)


class NN(Model):

    def __init__(self, X_train, y_train, X_val, y_val):
        super().__init__()
        self.epochs = 10
        self.checkpointer = ModelCheckpoint(filepath="best_model_weights.keras", verbose=1, save_best_only=True)
        self.max_log_y = max(numpy.max(numpy.log(y_train)), numpy.max(numpy.log(y_val)))
        self.__build_keras_model()
        self.fit(X_train, y_train, X_val, y_val)

    def __build_keras_model(self):
        self.model = Sequential()
        self.model.add(Dense(1000, kernel_initializer="uniform", input_dim=1183))
        self.model.add(Activation('relu'))
        self.model.add(Dense(500, kernel_initializer="uniform"))
        self.model.add(Activation('relu'))
        self.model.add(Dense(1))
        self.model.add(Activation('sigmoid'))

        self.model.compile(loss='mean_absolute_error', optimizer='adam')

    def _val_for_fit(self, val):
        val = numpy.log(val) / self.max_log_y
        return val

    def _val_for_pred(self, val):
        return numpy.exp(val * self.max_log_y)

    def fit(self, X_train, y_train, X_val, y_val):
        self.model.fit(X_train, self._val_for_fit(y_train),
                       validation_data=(X_val, self._val_for_fit(y_val)),
                       epochs=self.epochs, batch_size=128,
                       # callbacks=[self.checkpointer],
                       )
        # self.model.load_weights('best_model_weights.hdf5')
        print("Result on validation data: ", self.evaluate(X_val, y_val))

    def guess(self, features):
        result = self.model.predict(features).flatten()
        return self._val_for_pred(result)

In [6]:
# We moved Splitting to extra cell to ensure consistent train test split
import pickle
import numpy
numpy.random.seed(123)
from sklearn.preprocessing import OneHotEncoder
import sys
sys.setrecursionlimit(10000)

shuffle_data = False
one_hot_as_input = False
save_embeddings = True
saved_embeddings_fname = "embeddings.pickle"  # set save_embeddings to True to create this file

train_ratio = 0.9
f = open('feature_train_data.pickle', 'rb')
(X, y) = pickle.load(f)
num_records = len(X)
train_size = int(train_ratio * num_records)

if shuffle_data:
    print("Using shuffled data")
    sh = numpy.arange(X.shape[0])
    numpy.random.shuffle(sh)
    X = X[sh]
    y = y[sh]
X_train = X[:train_size]
X_val = X[train_size:]
y_train = y[:train_size]
y_val = y[train_size:]

def sample(X, y, n):
    '''random samples'''
    num_row = X.shape[0]
    indices = numpy.random.randint(num_row, size=n)
    return X[indices, :], y[indices]


X_train, y_train = sample(X_train, y_train, 200000)  # Simulate data sparsity
print("Number of samples used for training: " + str(y_train.shape[0]))

# Pickle Train & Test Split
for filename, data in [('X_train.pickle', X_train), ('y_train.pickle', y_train), ('X_val.pickle', X_val), ('y_val.pickle', y_val)]:
  with open(filename, 'wb') as f:
      pickle.dump(data, f, -1)

X_train.shape

Number of samples used for training: 200000


(200000, 8)

In [ ]:
from pathlib import Path
from copy import deepcopy

# Load Train/Test Split
for filename in ['X_train.pickle', 'y_train.pickle', 'X_val.pickle', 'y_val.pickle']:
  with open(filename, 'rb') as f:
    globals()[filename[:-7]] = pickle.load(f)

if one_hot_as_input:
    print("Using one-hot encoding as input")
    enc = OneHotEncoder(sparse=False)
    enc.fit(X_train)
    X_train = enc.transform(X_train)
    X_val = enc.transform(X_val)

models = []

# ? Are they fitting an Ensemble because latter they calc avg over those five models
print("Fitting NN_with_EntityEmbedding...")
for i in range(5):
    model = NN_with_EntityEmbedding(X_train, y_train, X_val, y_val)
    prediction_val = model.guess(X_val)
    prediction_train = model.guess(X_train)
    # dump to file
    path = Path(f"Predictions/NN_embed")
    path.mkdir(parents=True, exist_ok=True)
    with open(path / f"val_{i}.pickle", 'wb') as f:
      pickle.dump(prediction_val, f, -1)
    with open(path / f"train_{i}.pickle", 'wb') as f:
      pickle.dump(prediction_train, f, -1)
    models.append(model)

if save_embeddings:
    model = models[0].model
    store_embedding = model.get_layer('store_embedding').get_weights()[0]
    dow_embedding = model.get_layer('dow_embedding').get_weights()[0]
    year_embedding = model.get_layer('year_embedding').get_weights()[0]
    month_embedding = model.get_layer('month_embedding').get_weights()[0]
    day_embedding = model.get_layer('day_embedding').get_weights()[0]
    german_states_embedding = model.get_layer('state_embedding').get_weights()[0]
    with open(saved_embeddings_fname, 'wb') as f:
        pickle.dump([store_embedding, dow_embedding, year_embedding,
                     month_embedding, day_embedding, german_states_embedding], f, -1)

# print("Fitting NN...")
# for i in range(5):
#   model = NN(X_train, y_train, X_val, y_val)
#   prediction_val = model.guess(X_val)
#   prediction_train = model.guess(X_train)
#   # dump to file
#   path = Path(f"NN")
#   path.mkdir(parents=True, exist_ok=True)
#   with open(path / f"val_{i}.pickle", 'wb') as f:
#     pickle.dump(prediction_val, f, -1)
#   with open(path / f"train_{i}.pickle", 'wb') as f:
#     pickle.dump(prediction_train, f, -1)
#   models.append(model)

print("Fitting Traditional ML...")
for embeddings_as_input in [False, True]:
  if embeddings_as_input:
      print("Using learned embeddings as input")
      X_train_ = embed_features(X_train, saved_embeddings_fname)
      X_val_ = embed_features(X_val, saved_embeddings_fname)
  else:
    X_train_ = deepcopy(X_train)
    X_val_ = deepcopy(X_val)

  for model_name in ["MeanModel", "RF", "KNN", "XGBoost"]:
    model = globals()[model_name](X_train_, y_train, X_val_, y_val)
    prediction_val = model.guess(X_val_)
    prediction_train = model.guess(X_train_)
    # dumb to file
    path = Path("Predictions/" + model_name+("_embed" if embeddings_as_input else ""))
    path.mkdir(parents=True, exist_ok=True)
    with open(path / "val.pickle", 'wb') as f:
      pickle.dump(prediction_val, f, -1)
    with open(path / "train.pickle", 'wb') as f:
      pickle.dump(prediction_train, f, -1)
    models.append(model)

Fitting Traditional ML...
Result on validation data:  0.35751922035155137
Using learned embeddings as input
Result on validation data:  0.35751922035155137


# Now the Competition

In [11]:
import pandas as pd
from autogluon.tabular import TabularDataset, TabularPredictor
import pickle
from pathlib import Path

# Load Train/Test Split
for filename in ['X_train.pickle', 'y_train.pickle', 'X_val.pickle', 'y_val.pickle']:
  with open(filename, 'rb') as f:
    globals()[filename[:-7]] = pd.DataFrame(pickle.load(f))

X_train["Sales"] = y_train

df_val = TabularDataset(X_val)
df_train = TabularDataset(X_train)

# Mark categorical as categoricals
for i in [1, 2, 4, 5, 6, 7]: #Everything except promo and constant column are categoricals
    df_train[i] = df_train[i].astype("category")


df_train.head(100)

,0,1,2,3,4,5,6,7,Sales
0,0,467,5,0,0,1,18,7,5580
1,0,940,3,0,0,0,24,9,2953
2,0,272,5,0,0,0,10,6,6842
3,0,1022,0,1,0,9,21,2,8791
4,0,257,1,1,0,9,23,4,8830
...,...,...,...,...,...,...,...,...,...
95,0,1063,1,1,1,0,28,2,9559
96,0,478,4,1,0,6,18,6,3147
97,0,747,2,1,0,4,12,0,9057
98,0,943,2,1,0,5,27,6,5136


In [ ]:
for time_limit in [300, 600, 1800, 3600, 7200]:
  for quality in ["best_quality"]: 
    ag = TabularPredictor(label="Sales", path="tmp/1").fit(train_data=df_train, presets=quality, time_limit=time_limit)
    prediction_val = ag.predict(df_val)
    prediction_train = ag.predict(df_train)
    # dump
    path = Path(f"Predictions/AutoGluon_{time_limit}_{quality}")
    path.mkdir(parents=True, exist_ok=True)
    with open(path / "val.pickle", 'wb') as f:
      pickle.dump(prediction_val, f, -1)
    with open(path / "train.pickle", 'wb') as f:
      pickle.dump(prediction_train, f, -1)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.7
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.22631
CPU Count:          20
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       5.37 GB / 6.00 GB (89.5%)
Disk Space Avail:   197.36 GB / 951.65 GB (20.7%)
Presets specified: ['good_quality']
Using hyperparameters preset: hyperparameters='light'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if memory is small relative to the data size.
	You can avoid this risk by setting `save_bag_folds=True`.
DyStack is

In [21]:
from pathlib import Path
import pickle
import numpy as np


def _load_pickle(path: Path):
    with path.open("rb") as f:
        return pickle.load(f)


def _average_pickles(paths):
    data = [_load_pickle(p) for p in paths]
    return np.mean(np.stack(data, axis=0), axis=0)


def load_predictions(root: str):
    root_path = Path(root)
    results = {}

    for subdir in (p for p in root_path.iterdir() if p.is_dir()):

        # --- Case 1: sharded files exist ---
        train_shards = sorted(subdir.glob("train_*.pickle"))
        val_shards = sorted(subdir.glob("val_*.pickle"))

        if train_shards and val_shards:
            train_data = _average_pickles(train_shards)
            val_data = _average_pickles(val_shards)

        else:
            # --- Case 2: standard single files ---
            train_path = subdir / "train.pickle"
            val_path = subdir / "val.pickle"

            if not (train_path.exists() and val_path.exists()):
                continue

            train_data = _load_pickle(train_path)
            val_data = _load_pickle(val_path)

        results[subdir.name] = {
            "train": train_data,
            "val": val_data,
        }

    return results
predictions = load_predictions("Predictions")

In [22]:
for filename in ['y_train.pickle', 'y_val.pickle']:
  with open(filename, 'rb') as f:
    globals()[filename[:-7]] = pd.DataFrame(pickle.load(f))
for model, pred in predictions.items():
    print(model)
    train_mape = mape(y_train.iloc[:,0], pred["train"])
    test_mape = mape(y_val.iloc[:,0], pred["val"])
    print("Train:", train_mape)
    print("Test:", test_mape)

AutoGluon_1800_best_quality
Train: 0.04416106806594276
Test: 0.09873657703057198
AutoGluon_300_best_quality
Train: 0.05489121814871275
Test: 0.1035094158713566
AutoGluon_3600_best_quality
Train: 0.04648591092740764
Test: 0.09689775499898207
AutoGluon_600_best_quality
Train: 0.05015312420106384
Test: 0.10169996152989587
AutoGluon_7200_best_quality
Train: 0.046230513265724305
Test: 0.09796159528729156
KNN
Train: 1.4282142221561757e-05
Test: 0.2902914583664492
KNN_embed
Train: 4.3397421038566375e-16
Test: 0.11394754772746707
NN_embed
Train: 0.06335954469892272
Test: 0.09236548497257577
RF
Train: 0.05074499090912256
Test: 0.15813812160060575
RF_embed
Train: 0.02711571701120164
Test: 0.1048495134976635
XGBoost
Train: 0.15082793585988946
Test: 0.18450486642123445
XGBoost_embed
Train: 0.055163407786795904
Test: 0.11023669278357652


In [40]:
print(ag.feature_metadata.get_features(valid_special_types=["bool"]))

In [3]:
df_train.dtypes.to_frame("dtype")

,dtype
0,int64
1,category
2,category
3,int64
4,int64
5,int64
6,int64
7,int64
Sales,int64


In [2]:
df_train.describe()

,0,3,4,5,6,7,Sales
count,200000.0,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000
mean,0.0,0.443480,0.701385,5.232245,14.861520,5.158645,6920.723155
std,0.0,0.496796,0.709772,3.361613,8.896496,3.087252,3117.839189
min,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,46.000000
25%,0.0,0.000000,0.000000,2.000000,7.000000,2.000000,4820.000000
50%,0.0,0.000000,1.000000,5.000000,15.000000,6.000000,6329.000000
75%,0.0,1.000000,1.000000,8.000000,22.000000,8.000000,8316.000000
max,0.0,1.000000,2.000000,11.000000,30.000000,11.000000,38484.000000
